In [94]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,name,powerstats__intelligence,powerstats__strength,powerstats__speed,powerstats__durability,powerstats__power,powerstats__combat,Publisher,Gender,...,Super Breath,Wallcrawling,Vision - Night,Vision - Infrared,Vision - Heat,Vision - X-Ray,Vision - Thermal,meta_score,efficiency_delta,stability_index
0,179,Deathlok,67.854503,33.292252,30.244400,70.477368,40.627622,60.0,Marvel Comics,Male,...,False,False,False,True,False,False,False,0.273676,-1.577509,D
1,112,Box IV,49.574068,74.901682,24.369510,28.579236,10.886175,56.0,Marvel Comics,-,...,False,False,False,False,False,False,False,-1.517710,-8.211461,D
2,272,Hercules,64.491540,100.738794,45.815660,82.916665,90.097416,100.0,Marvel Comics,Male,...,False,False,False,False,False,False,False,-0.688044,8.189362,B
3,438,Phantom,-0.775359,NaN,-1.372187,-0.403532,-0.167523,-1.0,DC Comics,Male,...,False,False,False,False,False,False,False,0.138805,-16.458575,D
4,537,Spider-Woman IV,0.065715,0.078526,0.156320,-1.934220,0.859594,-1.0,Marvel Comics,Female,...,False,True,False,False,False,False,False,0.009787,-1.440590,C


In [95]:
#subtasks
subtask1 = df_train.Publisher.nunique()
subtask2 = df_train[df_train.Alignment == 'good'].Publisher.mode()[0]

df_1 = pd.DataFrame({
    "id": "GLOBAL",
    "subtaskID": ["task1", "task2"],
    "answer": [subtask1, subtask2]
})

df_1

,id,subtaskID,answer
0,GLOBAL,task1,23
1,GLOBAL,task2,Marvel Comics


In [96]:
df_train.Alignment.value_counts()

Alignment
good       323
bad        130
neutral     22
-            5
Name: count, dtype: int64

In [97]:
#simple preprocessing
df_train.Gender = df_train.Gender.map({"-": 0, "Male":1, "Female":2})
df_test.Gender = df_test.Gender.map({"-": 0, "Male":1, "Female":2})

df_train.Alignment = df_train.Alignment.map({"-":0, "bad":1, "neutral":2, "good":3})
df_test.Alignment = df_test.Alignment.map({"-":0, "bad":1, "neutral":2, "good":3})

df_train.head()

,id,name,powerstats__intelligence,powerstats__strength,powerstats__speed,powerstats__durability,powerstats__power,powerstats__combat,Publisher,Gender,...,Super Breath,Wallcrawling,Vision - Night,Vision - Infrared,Vision - Heat,Vision - X-Ray,Vision - Thermal,meta_score,efficiency_delta,stability_index
0,179,Deathlok,67.854503,33.292252,30.244400,70.477368,40.627622,60.0,Marvel Comics,1.0,...,False,False,False,True,False,False,False,0.273676,-1.577509,D
1,112,Box IV,49.574068,74.901682,24.369510,28.579236,10.886175,56.0,Marvel Comics,0.0,...,False,False,False,False,False,False,False,-1.517710,-8.211461,D
2,272,Hercules,64.491540,100.738794,45.815660,82.916665,90.097416,100.0,Marvel Comics,1.0,...,False,False,False,False,False,False,False,-0.688044,8.189362,B
3,438,Phantom,-0.775359,NaN,-1.372187,-0.403532,-0.167523,-1.0,DC Comics,1.0,...,False,False,False,False,False,False,False,0.138805,-16.458575,D
4,537,Spider-Woman IV,0.065715,0.078526,0.156320,-1.934220,0.859594,-1.0,Marvel Comics,2.0,...,False,True,False,False,False,False,False,0.009787,-1.440590,C


In [98]:
features = [col for col in df_test.columns if df_test[col].dtype != 'object' and col != 'id']
target1 = 'powerstats__combat'
target2 = 'Super Strength'

X = df_train[features]
y1 = df_train[target1]
y2 = df_train[target2]

X_final = df_test[features]

from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=3)
X = pd.DataFrame(imputer.fit_transform(X), columns=features)
X_final = pd.DataFrame(imputer.transform(X_final), columns=features)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.transform(X_final), columns=features)

X.head()

,powerstats__intelligence,powerstats__strength,powerstats__speed,powerstats__durability,powerstats__power,Gender,Height,Alignment,Weight,Agility,...,Vision - Microscopic,Super Breath,Wallcrawling,Vision - Night,Vision - Infrared,Vision - Heat,Vision - X-Ray,Vision - Thermal,meta_score,efficiency_delta
0,0.457236,-0.215075,-0.178864,0.611194,-0.417259,-0.494193,0.495736,0.677977,0.908004,1.282177,...,6.701425,-0.162551,-0.237666,-0.242281,5.357787,-0.162551,-0.168859,-0.203069,0.261687,-0.136519
1,-0.181884,0.988564,-0.386524,-0.619767,-1.259583,-2.562434,-1.577263,0.677977,-1.080324,-0.779923,...,-0.149222,-0.162551,-0.237666,-0.242281,-0.186644,-0.162551,-0.168859,-0.203069,-1.567800,-0.783143
2,0.339660,1.735956,0.371533,0.976658,0.983803,-0.494193,0.596454,0.677977,0.600800,-0.779923,...,-0.149222,-0.162551,-0.237666,-0.242281,-0.186644,-0.162551,-0.168859,-0.203069,-0.720487,0.815478
3,-1.942199,0.471936,-1.296416,-1.471277,-1.572641,-0.494193,-1.538440,0.677977,-1.144661,-0.779923,...,-0.149222,-0.162551,-0.237666,-0.242281,-0.186644,-0.162551,-0.168859,-0.203069,0.123947,-1.587006
4,-1.912793,-1.175851,-1.242388,-1.516248,-1.543551,1.574047,0.379697,-1.540458,-0.048205,1.282177,...,-0.149222,-0.162551,4.207577,-0.242281,-0.186644,-0.162551,-0.168859,-0.203069,-0.007814,-0.123173


In [99]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

model1 = RandomForestRegressor()
model2 = XGBClassifier()

print("Subtask 3: ", -cross_val_score(model1, X, y1, cv=5, scoring='neg_mean_absolute_error').mean())
print("Subtask 4: ", cross_val_score(model2, X, y2, cv=5, scoring='accuracy').mean())

Subtask 3:  13.062396039603962
Subtask 4:  0.8653465346534654


In [100]:
model1.fit(X, y1)
preds_1 = model1.predict(X_final)

model2.fit(X, y2)
preds_2 = model2.predict(X_final)

In [101]:
df_2 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": "task3",
    "answer": preds_1
})

df_2.head()

,id,subtaskID,answer
0,560,task3,77.16
1,626,task3,60.97
2,486,task3,57.45
3,45,task3,43.36
4,59,task3,90.35


In [102]:
df_3 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": "task4",
    "answer": preds_2
})

df_3.head()

,id,subtaskID,answer
0,560,task4,1
1,626,task4,0
2,486,task4,0
3,45,task4,0
4,59,task4,1


In [103]:
submission = pd.concat([df_1, df_2, df_3])
submission

,id,subtaskID,answer
0,GLOBAL,task1,23
1,GLOBAL,task2,Marvel Comics
0,560,task3,77.16
1,626,task3,60.97
2,486,task3,57.45
...,...,...,...
95,65,task4,1
96,141,task4,0
97,279,task4,0
98,472,task4,1


In [104]:
submission.to_csv("submission.csv", index=False)